# CelebA Hair Colour — ResultsTables only. Every number is read from `../CelebA/dr_mmd_flow/results/*.csv` and aggregatedin the house order: average the two arms within a (data seed, repeat), then within a dataseed, then mean ± sd across the three data seeds.**Design.** The CelebA hair-colour design of Dance et al., binary arm, outcome encoded by afrozen SD-VAE and decoded at 256 px. Default cell: `n_obs = 20000`, `M = 2000`, no-splitnuisance, observational initialisation, doubly robust target, projected Nystrom solver.**How to read a floor.** The oracle floor is a split-half of the held-out reference set, soit cannot be enlarged. SW2 and latent SW2 carry a count-matched twin that follows theparticle count and is capped at that half, so at `M = 2000` the flow is scored against twiceas many references as the floor and can print below it. FID is the legacy TorchVisionInception-v3 scorer, matching the DecFM convention; its count-matched column is in Section 2.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import analysis_tables as at

ALL = at.load("CelebA", estimator=None)
DR = ALL[ALL.estimator == "dr"]
BASE = DR[DR.init == "observed"]
DEFAULT = at.default_rows(ALL, dataset="CelebA", estimator=None)
DEF_DR = DEFAULT[DEFAULT.estimator == "dr"]
SEEDS = sorted(int(s) for s in ALL.data_seed.unique())
ARMS = sorted(int(a) for a in ALL.target_a.unique())
print(f"{len(ALL)} rows   stages {sorted(ALL.tag.unique())}")
print(f"seeds {SEEDS}   arms {ARMS}   estimators {sorted(ALL.estimator.unique())}")
print(f"n_obs {sorted(int(n) for n in ALL.n_obs.unique())}   "
      f"M {sorted(int(m) for m in ALL.n_particles_requested.unique())}   "
      f"dr_fit {sorted(ALL.dr_fit.unique())}   init {sorted(ALL.init.unique())}")
print(f"default cell: {len(DEFAULT)} rows = {len(SEEDS)} seeds x {len(ARMS)} arms "
      f"x 3 estimators   ({len(DEF_DR)} of them doubly robust)")

252 rows   stages ['est_crossfit', 'est_headline', 'est_init', 'est_nobs', 'est_particles']
seeds [0, 1, 2]   arms [0, 1]   estimators ['dr', 'ipw', 'plugin']
n_obs [5000, 10000, 15000, 20000]   M [250, 500, 1000, 2000]   dr_fit ['crossfit', 'nosplit']   init ['moment', 'normal', 'normal_raw', 'observed']
default cell: 18 rows = 3 seeds x 2 arms x 3 estimators   (6 of them doubly robust)


## 1. Headline

In [2]:
print(at.headline_table(DR, "CelebA", at.DECFM_CELEBA))

CelebA   (n_obs = 20000, M = 2000, seeds [0, 1, 2])

Method                SW2 ↓                FID ↓                Latent SW2 ↓
----------------------------------------------------------------------------
DecFM-Plugin          0.03272 ± 0.00183    11.497 ± 0.156       0.06220 ± 0.00071
DecFM-DR-I            0.03356 ± 0.00318    14.609 ± 0.525       0.06241 ± 0.00405
DecFM-DR-OT           0.03636 ± 0.00245    15.521 ± 1.307       0.06746 ± 0.00257
IMMD-Flow (ours)      0.02656 ± 0.00155    11.567 ± 0.085       0.05129 ± 0.00136

Ours: doubly robust target embedding, projected Nystrom, no-split nuisance, no-split autoencoder, P_Y initialisation.
FID is the legacy TorchVision Inception-v3 scorer, matching the reference convention.


The flow is lower than every reference row on both distributional columns: SW2 **0.02656**against the best reference 0.03272, latent SW2 **0.05129** against 0.06220. FID **11.567**sits just above DecFM-Plugin's 11.497 and below both DR variants. All four of our numberspool the two arms; Section 2 adds the floors.

## 2. Baselines and floors

In [3]:
ROWS = [("No flow (logged arm)", ["sw2_observed", "fid_legacy_observed",
                                  "fid_legacy_observed_half", "latent_sw2_initial"]),
        ("VAE round-trip",       ["sw2_roundtrip", "fid_legacy_roundtrip",
                                  "fid_legacy_roundtrip_half", None]),
        ("IMMD-Flow (ours)",   ["sw2_flow", "fid_legacy_flow",
                                  "fid_legacy_flow_half", "latent_sw2_flow"]),
        ("Split-half floor",     ["sw2_floor", None,
                                  "fid_legacy_floor", "latent_sw2_floor"])]
print(at.plain_table(
    [(label, [at.summarise(DEF_DR, c) if c else None for c in columns])
     for label, columns in ROWS],
    ["SW2", "FID", "FID (1000 vs 1000)", "Latent SW2"], first_column="Row", label_width=22,
    title=f"CelebA   (n_obs = 20000, M = 2000, seeds {SEEDS}, arms pooled)",
    note="No flow = the logged rows of the target arm. VAE round-trip = those same images\n"
         "encoded and decoded once, with no flow. The floor row is a 1000-vs-1000 split of\n"
         "the reference set, so only the `1000 vs 1000` FID column compares with it at\n"
         "equal counts; the SW2 and Latent SW2 rows above it are scored at 2000."))
initial, _ = at.summarise(DEF_DR, "latent_sw2_initial")
floor, _ = at.summarise(DEF_DR, "latent_sw2_floor")
print(f"\nUnflowed logged arm over the latent floor: {initial / floor:.3f}x")

CelebA   (n_obs = 20000, M = 2000, seeds [0, 1, 2], arms pooled)

Row                   SW2 ↓                FID ↓                FID (1000 vs 1000) ↓ Latent SW2 ↓
-------------------------------------------------------------------------------------------------
No flow (logged arm)  0.04960 ± 0.00130    11.082 ± 0.129       19.434 ± 0.380       0.07703 ± 0.00322
VAE round-trip        0.04974 ± 0.00129    11.499 ± 0.102       19.863 ± 0.455       --
IMMD-Flow (ours)      0.02656 ± 0.00155    11.567 ± 0.085       19.957 ± 0.242       0.05129 ± 0.00136
Split-half floor      0.04001 ± 0.00240    --                   19.086 ± 0.269       0.07744 ± 0.00164

No flow = the logged rows of the target arm. VAE round-trip = those same images
encoded and decoded once, with no flow. The floor row is a 1000-vs-1000 split of
the reference set, so only the `1000 vs 1000` FID column compares with it at
equal counts; the SW2 and Latent SW2 rows above it are scored at 2000.

Unflowed logged arm over the l

Before any flow the logged arm is at SW2 0.04960 and latent SW2 0.07703; the flow moves themto 0.02656 and 0.05129. Encode-and-decode alone costs almost nothing in SW2 (round-trip0.04974), and at equal counts FID reads 19.434 logged and 19.957 flowed against a 19.086floor. The latent floor is 0.07744 and the unflowed arm already sits at **0.995x** of it, solatent SW2 has little resolving power here and is readable only with that floor beside it.

## 3. Per arm

In [4]:
print(f"  {'arm':>3s} {'logged n':>9s} {'ESS on arm':>11s} {'SW2 logged':>11s} "
      f"{'SW2 flow':>9s} {'FID flow':>9s} {'Latent logged':>14s} {'Latent flow':>12s}")
for a in ARMS:
    g = DEF_DR[DEF_DR.target_a == a]
    n, ess, obs, flow, fid, lat0, lat = [
        at.summarise(g, c)[0] for c in ("n_on_arm", "ess_on_arm", "sw2_observed", "sw2_flow",
                                        "fid_legacy_flow", "latent_sw2_initial",
                                        "latent_sw2_flow")]
    print(f"  {a:3d} {n:9.0f} {ess:11.0f} {obs:11.5f} {flow:9.5f} {fid:9.3f} "
          f"{lat0:14.5f} {lat:12.5f}")

  arm  logged n  ESS on arm  SW2 logged  SW2 flow  FID flow  Latent logged  Latent flow
    0     11000        9600     0.05400   0.02726    10.587        0.07999      0.05078
    1      9000        4800     0.04520   0.02586    12.548        0.07408      0.05179


The two arms are not interchangeable: arm 0 carries 11000 logged rows and an on-arm effectivesample size of 9600, arm 1 carries 9000 and 4800. Arm 1 starts closer in pixel SW2 (0.04520against 0.05400) and ends closer (0.02586 against 0.02726), while its FID is the worse of thetwo (12.548 against 10.587).

## 4. Target-embedding estimator

In [5]:
print(at.estimator_table(DEFAULT, "CelebA"))
print()
KEYS = ["tag", "data_seed", "repeat", "target_a", "n_obs", "n_particles_requested",
        "dr_fit", "init"]
wide = {e: ALL[ALL.estimator == e].set_index(KEYS).sort_index()
        for e in ("plugin", "ipw", "dr")}
print(f"Paired per-run differences over all {len(wide['dr'])} runs:")
for label, column in at.METRICS["CelebA"]:
    ipw = (wide["ipw"][column] - wide["dr"][column]).abs().max()
    plugin = (wide["plugin"][column] - wide["dr"][column]).abs().max()
    print(f"  {label:<12} max |IPW - DR| = {ipw:.2e}   max |plug-in - DR| = {plugin:.2e}")

CelebA   target embedding estimator

Target embedding          SW2 ↓                FID ↓                Latent SW2 ↓
--------------------------------------------------------------------------------
Plug-in (CME only)        0.02747 ± 0.00123    11.861 ± 0.074       0.06604 ± 0.00139
IPW (ratios only)         0.02656 ± 0.00155    11.567 ± 0.085       0.05129 ± 0.00136
Doubly robust (default)   0.02656 ± 0.00155    11.567 ± 0.085       0.05129 ± 0.00136

One logged sample, one nuisance solve, one set of initial particles and one flow budget per run;
only the target weight vector differs.



Paired per-run differences over all 84 runs:
  SW2          max |IPW - DR| = 5.39e-04   max |plug-in - DR| = 2.36e-02
  FID          max |IPW - DR| = 1.74e-02   max |plug-in - DR| = 7.69e+00
  Latent SW2   max |IPW - DR| = 9.95e-04   max |plug-in - DR| = 2.76e-01


IPW and DR print the same value in every column (SW2 0.02656, FID 11.567, latent SW20.05129), and the plug-in row is worse on all three (0.02747, 11.861, 0.06604). The paireddifferences agree run by run: 5.39e-04 at most in SW2 between IPW and DR, against 2.36e-02between plug-in and DR. The axis separates the plug-in embedding here and nothing else.

## 5. AblationsOne helper, four sweeps. The first two carry the count-matched floors, because those movealong the row; the last two sit at the default cell.

In [6]:
FLOW = [("SW2", "sw2_flow"), ("FID", "fid_legacy_flow"), ("Latent SW2", "latent_sw2_flow")]
WITH_FLOOR = [("SW2 flow", "sw2_flow"), ("SW2 floor", "sw2_floor_matched"),
              ("FID", "fid_legacy_flow"), ("Latent flow", "latent_sw2_flow"),
              ("Latent floor", "latent_sw2_floor_matched")]
NOBS = [("SW2 logged", "sw2_observed")] + WITH_FLOOR


def sweep(frame, by, order, labels, title, metrics=FLOW, first_column="Setting"):
    rows = [(labels[value], [at.summarise(frame[frame[by] == value], column)
                             for _, column in metrics]) for value in order]
    print(at.plain_table(rows, [label for label, _ in metrics], title=title,
                         first_column=first_column, label_width=22, cell_width=19))
    print()


NOSPLIT = BASE[BASE.dr_fit == "nosplit"]
sweep(NOSPLIT[NOSPLIT.n_particles_requested == 2000], "n_obs", [5000, 10000, 15000, 20000],
      {n: f"n_obs = {n}" for n in (5000, 10000, 15000, 20000)},
      "Observations   (M = 2000, no-split)", metrics=NOBS)
sweep(NOSPLIT[NOSPLIT.n_obs == 20000], "n_particles_requested", [250, 500, 1000, 2000],
      {m: f"M = {m}" for m in (250, 500, 1000, 2000)},
      "Particles   (n_obs = 20000, no-split)", metrics=WITH_FLOOR)
sweep(BASE[(BASE.n_obs == 20000) & (BASE.n_particles_requested == 2000)], "dr_fit",
      ["nosplit", "crossfit"], {"nosplit": "No-split", "crossfit": "Cross-fitted"},
      "Nuisance fitting   (n_obs = 20000, M = 2000)")
sweep(DR[(DR.dr_fit == "nosplit") & (DR.n_obs == 20000)
         & (DR.n_particles_requested == 2000)], "init",
      ["observed", "moment", "normal", "normal_raw"],
      {"observed": "Observed (P_Y)", "moment": "Moment-matched",
       "normal": "Normal (standardised)", "normal_raw": "Normal (raw space)"},
      "Initialisation   (n_obs = 20000, M = 2000)")

Observations   (M = 2000, no-split)

Setting               SW2 logged ↓       SW2 flow ↓         SW2 floor ↓        FID ↓              Latent flow ↓      Latent floor ↓
-----------------------------------------------------------------------------------------------------------------------------------
n_obs = 5000          0.05557 ± 0.00561  0.02996 ± 0.00245  0.04074 ± 0.00216  11.589 ± 0.176     0.05674 ± 0.00145  0.07801 ± 0.00144
n_obs = 10000         0.05576 ± 0.00501  0.02954 ± 0.00128  0.04565 ± 0.00342  11.446 ± 0.129     0.05444 ± 0.00060  0.08038 ± 0.00242
n_obs = 15000         0.05503 ± 0.00504  0.02705 ± 0.00138  0.04259 ± 0.00128  11.593 ± 0.151     0.05160 ± 0.00113  0.07967 ± 0.00061
n_obs = 20000         0.04960 ± 0.00130  0.02656 ± 0.00155  0.04001 ± 0.00240  11.567 ± 0.085     0.05129 ± 0.00136  0.07744 ± 0.00164



Particles   (n_obs = 20000, no-split)

Setting               SW2 flow ↓         SW2 floor ↓        FID ↓              Latent flow ↓      Latent floor ↓
----------------------------------------------------------------------------------------------------------------
M = 250               0.07024 ± 0.00169  0.08079 ± 0.00708  75.714 ± 0.883     0.13115 ± 0.00103  0.15177 ± 0.00247
M = 500               0.04838 ± 0.00138  0.05685 ± 0.00188  37.813 ± 0.705     0.09404 ± 0.00087  0.10897 ± 0.00167
M = 1000              0.03555 ± 0.00222  0.04001 ± 0.00240  19.226 ± 0.027     0.06950 ± 0.00181  0.07744 ± 0.00164
M = 2000              0.02656 ± 0.00155  0.04001 ± 0.00240  11.567 ± 0.085     0.05129 ± 0.00136  0.07744 ± 0.00164

Nuisance fitting   (n_obs = 20000, M = 2000)

Setting               SW2 ↓              FID ↓              Latent SW2 ↓
------------------------------------------------------------------------
No-split              0.02656 ± 0.00155  11.567 ± 0.085     0.05129 ± 0.00136


Initialisation   (n_obs = 20000, M = 2000)

Setting               SW2 ↓              FID ↓              Latent SW2 ↓
------------------------------------------------------------------------
Observed (P_Y)        0.02656 ± 0.00155  11.567 ± 0.085     0.05129 ± 0.00136
Moment-matched        0.03780 ± 0.00040  211.398 ± 1.631    0.05131 ± 0.00153
Normal (standardised) 0.08618 ± 0.00134  292.896 ± 1.772    0.05417 ± 0.00158
Normal (raw space)    0.12753 ± 0.00144  314.482 ± 1.147    0.07028 ± 0.00112



**Observations.** SW2 falls 0.02996 → 0.02656 and latent SW2 0.05674 → 0.05129 over afourfold increase in n_obs, with FID between 11.446 and 11.593. The logged baseline is notconstant down the ladder (0.05557, 0.05576, 0.05503, 0.04960) and neither are the floors(0.04074, 0.04565, 0.04259, 0.04001): the reference count is fixed, but the reference imagesare redrawn for each n_obs.**Particles.** Every column falls with M: SW2 0.07024 → 0.02656, FID 75.714 → 11.567, latentSW2 0.13115 → 0.05129. The matched floors fall with them as far as M = 1000 (0.08079 →0.04001) and then stop, since the reference half cannot be enlarged. Up to M = 1000 each flowvalue is below its own matched floor; at M = 2000 flow and floor are not at the same count.**Nuisance fitting.** The two fits agree in all three columns to the printed precision(0.02656, 11.567, 0.05129); only the seed spread differs, in the fifth decimal of SW2.**Initialisation.** The observational start is far ahead on the pixel columns: SW2 0.02656against 0.03780, 0.08618 and 0.12753, FID 11.567 against 211.398, 292.896 and 314.482.Latent SW2 barely moves across the first three starts (0.05129, 0.05131, 0.05417) and onlythe raw-space normal costs it anything (0.07028).

## 6. Coverage

In [7]:
print(at.plain_table(
    [(tag, [(len(g), 0.0), (len(g) / 6, 0.0)]) for tag, g in ALL.groupby("tag", observed=True)],
    ["rows", "cells"], first_column="Launcher stage", label_width=18, digits=0, arrow=False,
    title="CelebA   what is on disk",
    note="cells = rows / (2 arms x 3 estimators); 3 seeds each."))
print(f"\ntotal {len(ALL)} rows, all five stages complete at {len(SEEDS)} seeds.")

CelebA   what is on disk

Launcher stage    rows                 cells
--------------------------------------------
est_crossfit      72 ± 0               12 ± 0
est_headline      18 ± 0               3 ± 0
est_init          54 ± 0               9 ± 0
est_nobs          54 ± 0               9 ± 0
est_particles     54 ± 0               9 ± 0

cells = rows / (2 arms x 3 estimators); 3 seeds each.

total 252 rows, all five stages complete at 3 seeds.
